# BLCA Classifier

In [29]:
import os
import sys
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, QuantileTransformer, RobustScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.base import BaseEstimator
from sklearn.metrics import log_loss, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

import umap

# Data Loading

Load and transform the data

In [30]:
pd.set_option("display.max_columns", 500)

In [31]:
#df = pd.read_csv("/drive3/kjliu1/cfRNA/NovaSeqX594/DNA_modified/output/TMM_NONE/TMM_NONE_gene_log2nx.txt", sep='\t')
df = pd.read_csv("/drive3/kjliu1/cfRNA/NovaSeqX526/output/TMM_NONE/TMM_NONE_gene_log2nx.txt", sep='\t')

sample_info = pd.read_csv("/drive3/kjliu1/python_server/uRAGs_selector/Tumor_CTR_sampleinfo_uRAGs.txt", sep='\t')


#sex_diff_gene_df = pd.read_csv("/drive3/kjliu1/cfRNA/sex_difference_CTR_urine/sex_difference_CTR_urine_urag/sex_diff_uRAG_pvalue.txt", sep='\t')
sex_chr_gene_df = pd.read_csv("/drive3/kjliu1/python_server/sex_chromosome_gene_info_annotated.txt", sep='\t')

public_diff_gene_df = pd.read_csv("/drive3/kjliu1/cfRNA/TCGA_enrichment_gene_set/TCGA_enrichment_geneset_LUTS_uRAGs/BLCA/TCGABLCAvsControl_DEG_pvalue_LFC3.txt", sep='\t')
blood_gene_df = pd.read_csv("/drive3/kjliu1/cfRNA/TWIST_uRAREseq_selector_design/GTEX_RARE/GTEX_tmm_avg_perc_L0_P20.txt", sep='\t')

sex_diff_gene_df = pd.read_csv("/drive3/kjliu1/cfRNA/TWIST_uRAREseq_selector_design/LUTS_reference_uRAG/LUTS_reference_sex/output/Sex_diff_LUTS_uRAG_pvalue.txt", sep='\t')
#stone_gene_df = pd.read_csv("/drive3/kjliu1/cfRNA/TWIST_uRAREseq_selector_design/LUTS_reference_uRAG/LUTS_reference_sex/output/Sex_diff_LUTS_uRAG_pvalue.txt", sep='\t')

removeLM_gene_df = pd.read_csv("/drive3/kjliu1/python_server/uRAGs_selector/remove_LM22_genes.txt", sep='\t')


In [ ]:
# transform df
# make columns "gene_id" and add a "sample_id" 
# make each row a sample
df = df.T
df.columns = df.iloc[0]
df = df.drop(df.index[0])
df = df.reset_index()
df = df.rename(columns={"index": "sample_id"})
df.columns = df.iloc[1, :]
# remove row 1 annd 2
df = df.drop(df.index[0])
df = df.drop(df.index[0])
df = df.reset_index()
# rename gene_symbol to "sample_id"
df = df.rename(columns={"gene_symbol": "sample_id"})
# drop first 2 columns
df = df.iloc[:, 1:]

df_backup = df.copy()

print(df)

In [ ]:


# remove values in gene_symbol of sex_diff_gene_df from df columns
remove_genes = sex_diff_gene_df['gene_symbol'].tolist()
# Remove these columns from df
df = df.drop(columns=[col for col in remove_genes if col in df.columns])

# remove values in gene_symbol of sex_chromosome_gene_df from df columns
remove_genes_sex = sex_chr_gene_df['gene_symbol'].tolist()
# Remove these columns from df
df = df.drop(columns=[col for col in remove_genes_sex if col in df.columns])

# Keep genes that are Differentially expressed in UROMOL vs TCGA
public_keep_genes = public_diff_gene_df['gene_symbol'].tolist()
# Include the sample ID column
columns_to_keep = ['sample_id'] + [col for col in public_keep_genes if col in df.columns]
# Keep the DEG UROMOL vs TCGA DEG from df 
df = df[columns_to_keep]

# Keep genes that are in geneinfo from Monica
#geneinfo_genes = geneinfo_df['gene_symbol'].tolist()
# Include the sample ID column
#columns_to_keep2 = ['sample_id'] + [col for col in geneinfo_genes if col in df.columns]
# Keep the DEG UROMOL vs TCGA DEG from df 
#df = df[columns_to_keep2]

# Keep genes that are in not express in reference LUTS
#reference_genes = reference_gene_df['gene_symbol'].tolist()
# Include the sample ID column
#columns_to_keep_reference = ['sample_id'] + [col for col in reference_genes if col in df.columns]
# Keep the genes that are RARE in reference
#df = df[columns_to_keep_reference]


# Keep genes that are RARE in blood
blood_genes = blood_gene_df['gene_symbol'].tolist()
# Include the sample ID column
columns_to_keep_blood = ['sample_id'] + [col for col in blood_genes if col in df.columns]
# Keep the genes that are RARE in blood
df = df[columns_to_keep_blood]


# remove values in gene_symbol of stone genes from df columns
#remove_genes_stone = stone_gene_df['gene_symbol'].tolist()
# Remove these columns from df
#df = df.drop(columns=[col for col in remove_genes_stone if col in df.columns])

# Keep genes that are RARE in blood
#stone_genes = stone_gene_df['gene_symbol'].tolist()
# Include the sample ID column
#columns_to_keep_stone = ['sample_id'] + [col for col in stone_genes if col in df.columns]
# Keep the genes that are RARE in blood
#df = df[columns_to_keep_stone]

# remove values in gene_symbol of preBCG samples that are DNA negative from df columns
remove_LM = removeLM_gene_df['gene_symbol'].tolist()
# Remove these columns from df
df = df.drop(columns=[col for col in remove_LM if col in df.columns])

#can't add CRH or ANXA10 because those are aready in the list
#cols_to_save = ["sample_id","TERT", "IGF2", "IGFBP5", "CDK1","UPK1B", "CEACAM21", "CEACAM7"]
#cols_to_save = ["sample_id","TERT", "IGF2", "IGFBP5", "CDK1","UPK1B"]
cols_to_save = ["sample_id","IGF2", "TERT"]
#cols_to_save = ["sample_id", "TERT"]
add_back_cols = df_backup.loc[:, cols_to_save]
df = pd.merge(df, add_back_cols, how="left", on="sample_id")

print(df)


In [ ]:
print(df)

In [7]:
sample_info = sample_info.rename(columns={"Sample": "sample_id"})
sample_info = sample_info.loc[:, ["sample_id", "Sample_type"]]

In [8]:
df = df.merge(sample_info, on="sample_id", how="inner")

In [ ]:
# encode columns "Sample_type", save dictionary of encoding
#le = LabelEncoder()
#le.fit(df["Sample_type"])
#df["Sample_type"] = le.transform(df["Sample_type"])
#label_to_key = dict(zip(le.classes_, le.transform(le.classes_)))
#key_to_label = dict(zip(le.transform(le.classes_), le.classes_))

# set MIBC to 1 and NMIBC to 0
df["Sample_type"] = df["Sample_type"].replace({"BLCA": 1, "Control": 0})
key_to_label = {0: "Control", 1: "BLCA"}

In [ ]:
key_to_label

In [11]:
X_df = df.iloc[:, 1:-1]
y_df = df.iloc[:, -1]
X = X_df.values
y = y_df.values

id_df = df.iloc[:, 0]
id_values = id_df.values

# Logistic regression with Elastic Net

In [12]:
#remove L1 in penalty so it doesn't choose top features. Use all. 
def fit_model(X, y, cv=5):
    """Fit a logistic regression model using LogisticRegressionCV.
    """
    Cs = np.logspace(-5, 4, 100)
    model = LogisticRegressionCV(Cs=Cs, cv=cv, scoring="neg_log_loss", max_iter=10000, class_weight="balanced",l1_ratios=[0.1],penalty="elasticnet", solver="saga", n_jobs=36, random_state=42)
    model.fit(X, y)
    return model

In [13]:
# Retrieve the best l1_ratios
#best_l1_ratios = model.l1_ratio_

#print("Best l1_ratios:", best_l1_ratios)

In [ ]:
# do a 10 fold CV split over X and y
all_X_test = []
all_y_test = []
all_id_test = []
all_y_pred = []
best_params = []
models = []
scalers = []

cv = StratifiedKFold(n_splits=10, random_state=42, shuffle=True)

for train_index, test_index in cv.split(X, y):
    id_train, id_test = id_values[train_index], id_values[test_index]
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    model = fit_model(X_train, y_train, cv=5)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    print(f"log loss: {log_loss(y_test, y_pred_proba)}")
    print(confusion_matrix(y_test, y_pred))

    # get the best params
    best_params.append(model.C_)
    print(f"best params: {model.C_}")
    models.append(model)
    all_y_test.extend(y_test)
    all_id_test.extend(id_test)
    all_X_test.append(X_test)
    # just append the probabilities of the positive class
    all_y_pred.extend(y_pred_proba[:, 1])
    scalers.append(scaler)
    # get nonzero coefficients
    nonzero = np.where(model.coef_ != 0)[1]
    X_train = X_train[:, nonzero]
    X_test = X_test[:, nonzero]


In [ ]:
# print the overall ROC curve
fpr, tpr, _ = roc_curve(all_y_test, all_y_pred)
roc_auc = auc(fpr, tpr)
# compute CI using delongs method
plt.figure()
lw = 2
plt.plot(fpr, tpr, color='darkorange',
         lw=lw, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC - BLCA vs Control')
plt.legend(loc="lower right")
plt.show()

In [ ]:
#write down result_df to correlate sample ID with prediction value

results_df_sample = pd.DataFrame({"pred": all_y_pred, "true": all_y_test, "id": all_id_test})
print(results_df_sample)

In [17]:
results_df_sample.to_csv("BLCA_Elasticnet_uRAGs_predictionvalue.csv", index=False)

# Plot the most important features

In [ ]:
# how many non-zero coefficients are there?
for model in models:
    print(np.sum(model.coef_ != 0))

In [ ]:
# plot the coefficient values
for model in models:
    # for each model, plot x = order of coefficient, descending by absolute value
    # y = value of coefficient
    # plot as a line plot
    abscoef = np.abs(model.coef_).flatten()
    order = np.argsort(abscoef)[::-1]
    plt.plot(abscoef[order])
plt.xlabel("Coefficient number")
plt.ylabel("Coefficient value")
plt.show()

In [20]:
# compute the SHAP values
shap_values = []
for idx, model in enumerate(models):
    X_test = all_X_test[idx]
    explainer = shap.LinearExplainer(model, X_test)
    shap_values.append(explainer.shap_values(X_test))
shap_values = np.concatenate(shap_values, axis=0)

In [ ]:
# plot the SHAP summary plot
shap.summary_plot(shap_values, np.concatenate(all_X_test, axis=0), feature_names=X_df.columns)

# Tree classifier

# Save a table of the features with non-zero coefficients

In [22]:
# get the nonzero coefficients
# get the coefficient value
nonzero_coefs = []
coef_values = []
for model in models:
    nonzero_coefs.append(np.where(model.coef_ != 0)[1])
    coef_values.append(model.coef_[0, nonzero_coefs[-1]])

# get the names of the nonzero coefficents
nonzero_names = []
for nonzero in nonzero_coefs:
    nonzero_names.append(X_df.columns[nonzero])

# count the number of times each name appears
nonzero_counts = {}
all_nonzero_names = np.concatenate(nonzero_names)
all_nonzero_coefs = np.concatenate(coef_values)
for idx, name in enumerate(all_nonzero_names):
    coef = all_nonzero_coefs[idx]
    if name in nonzero_counts:
        nonzero_counts[name] = {"gene": name, "num_models_non_zero": nonzero_counts[name]["num_models_non_zero"] + 1, "coef": nonzero_counts[name]['coef'] + [coef]}
    else:
        nonzero_counts[name] = {"gene": name, "num_models_non_zero": 1, "coef": [coef]}

In [23]:
nonzero_counts = pd.DataFrame(nonzero_counts).T.reset_index(drop=True)

In [24]:
nonzero_counts['mean_coef'] = nonzero_counts['coef'].apply(lambda x: np.mean(x))
nonzero_counts['var_coef'] = nonzero_counts['coef'].apply(lambda x: np.var(x))
nonzero_counts['abs_coef'] = nonzero_counts['mean_coef'].apply(lambda x: np.abs(x))

In [25]:
nonzero_counts.sort_values(["num_models_non_zero", "abs_coef"], ascending=False, inplace=True)

In [ ]:
nonzero_counts['num_models_non_zero'].value_counts()

In [ ]:
nonzero_counts.head(50)

In [28]:
# drop coef column
nonzero_counts = nonzero_counts.drop(columns=["coef"])
# save to file
nonzero_counts.to_csv("BLCA_elasticnet_nonzero_genes_uRAGs.csv", index=False)